# SQL Loop Basic

Demo básica para entender la evolución de **Prompting → Looping**.

**Objetivo:** optimizar una consulta SQL hasta alcanzar una mejora de rendimiento definida.

## 1. Setup

Validamos el entorno Python y las dependencias necesarias para ejecutar la demo.

In [132]:
import sys
import duckdb
import time
from pathlib import Path

print("Python :", sys.version.split()[0])
print("DuckDB :", duckdb.__version__)
print("Kernel :", sys.executable)

Python : 3.13.9
DuckDB : 1.5.5
Kernel : c:\Dev\Proyectos\Code Scripts\Agentes_IA\Looping\sql-loop\.venv\Scripts\python.exe


## 2. Base de datos

Usaremos **DuckDB** como base de datos local para mantener la demo simple y reproducible.

In [133]:
DB_PATH = Path("../data/sql_loop.duckdb")

con = duckdb.connect(str(DB_PATH))

print(f"Database: {DB_PATH.resolve()}")
print("DuckDB connected ✓")

Database: C:\Dev\Proyectos\Code Scripts\Agentes_IA\Looping\sql-loop\data\sql_loop.duckdb
DuckDB connected ✓


### Validación de conexión

Ejecutamos una consulta simple para comprobar que DuckDB responde correctamente.

In [134]:
result = con.execute("""
    SELECT
        1 AS id,
        'SQL Loop Basic' AS demo,
        CURRENT_TIMESTAMP AS executed_at
""").fetchdf()

result

,id,demo,executed_at
0,1,SQL Loop Basic,2026-08-16 23:03:37.941498-05:00


## 3. Dataset sintético — Tarjetas de Crédito

Simulamos un escenario simplificado de tarjetas de crédito con tres entidades:

- **Customers:** clientes.
- **Cards:** tarjetas asociadas a los clientes.
- **Transactions:** consumos realizados con las tarjetas.

El objetivo será analizar clientes Premium con tarjetas activas y transacciones aprobadas.

In [135]:
con.execute("""
    CREATE OR REPLACE TABLE customers AS
    SELECT
        i AS customer_id,
        'Customer_' || i AS customer_name,
        CASE
            WHEN i % 10 = 0 THEN 'PREMIUM'
            WHEN i % 3 = 0 THEN 'BUSINESS'
            ELSE 'STANDARD'
        END AS segment
    FROM range(1, 100001) t(i)
""")

count = con.execute(
    "SELECT COUNT(*) FROM customers"
).fetchone()[0]

print(f"Customers: {count:,}")

Customers: 100,000


In [136]:
con.execute("""
    CREATE OR REPLACE TABLE cards AS
    SELECT
        i AS card_id,
        1 + (i % 100000) AS customer_id,
        CASE
            WHEN i % 3 = 0 THEN 'PLATINUM'
            WHEN i % 3 = 1 THEN 'GOLD'
            ELSE 'CLASSIC'
        END AS card_type,
        CASE
            WHEN i % 10 = 0 THEN 'BLOCKED'
            ELSE 'ACTIVE'
        END AS status
    FROM range(1, 150001) t(i)
""")

count = con.execute(
    "SELECT COUNT(*) FROM cards"
).fetchone()[0]

print(f"Cards: {count:,}")

Cards: 150,000


In [137]:
con.execute("""
    CREATE OR REPLACE TABLE transactions AS
    SELECT
        i AS transaction_id,

        1 + (i % 150000) AS card_id,

        DATE '2024-01-01'
            + CAST(hash(i * 17) % 730 AS INTEGER)
            AS transaction_date,

        CAST(
            10 + (hash(i * 31) % 1990)
            AS DECIMAL(12,2)
        ) AS amount,

        CASE
            WHEN hash(i * 47) % 10 < 8 THEN 'APPROVED'
            WHEN hash(i * 47) % 10 = 8 THEN 'DECLINED'
            ELSE 'REVERSED'
        END AS transaction_status

    FROM range(1, 20000001) t(i)
""")

count = con.execute(
    "SELECT COUNT(*) FROM transactions"
).fetchone()[0]

print(f"Transactions: {count:,}")

Transactions: 20,000,000


### Validación del dataset

Verificamos el volumen de información generado para la demo.

In [138]:
summary = con.execute("""
    SELECT 'customers' AS table_name, COUNT(*) AS rows
    FROM customers

    UNION ALL

    SELECT 'cards', COUNT(*)
    FROM cards

    UNION ALL

    SELECT 'transactions', COUNT(*)
    FROM transactions
""").fetchdf()

summary

,table_name,rows
0,customers,100000
1,cards,150000
2,transactions,20000000


In [139]:
validation = con.execute("""
    SELECT
        c.segment,
        ca.status AS card_status,
        t.transaction_status,
        COUNT(*) AS transactions
    FROM customers c
    JOIN cards ca
        ON c.customer_id = ca.customer_id
    JOIN transactions t
        ON ca.card_id = t.card_id
    WHERE c.segment = 'PREMIUM'
    GROUP BY
        c.segment,
        ca.status,
        t.transaction_status
    ORDER BY transactions DESC
""").fetchdf()

validation

,segment,card_status,transaction_status,transactions
0,PREMIUM,ACTIVE,APPROVED,1599698
1,PREMIUM,ACTIVE,DECLINED,200342
2,PREMIUM,ACTIVE,REVERSED,199960


## 4. Query inicial y Baseline

### Caso de negocio

Queremos identificar los **20 clientes Premium con mayor consumo durante 2025**, considerando únicamente:

- tarjetas activas;
- transacciones aprobadas.

La consulta inicial es funcional, pero contiene oportunidades de optimización que deberá descubrir el modelo.

In [140]:
slow_query = """
SELECT
    c.customer_id,
    c.customer_name,
    SUM(t.amount) AS total_spent,
    COUNT(t.transaction_id) AS total_transactions
FROM customers c
JOIN cards ca
    ON c.customer_id = ca.customer_id
JOIN transactions t
    ON ca.card_id = t.card_id
WHERE
    LOWER(c.segment) = 'premium'
    AND UPPER(ca.status) = 'ACTIVE'
    AND UPPER(t.transaction_status) = 'APPROVED'
    AND STRFTIME(t.transaction_date, '%Y') = '2025'
    AND t.amount > (
        SELECT AVG(amount)
        FROM transactions
        WHERE STRFTIME(transaction_date, '%Y') = '2025'
    )
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    total_spent DESC
LIMIT 20;
"""

print(slow_query)


SELECT
    c.customer_id,
    c.customer_name,
    SUM(t.amount) AS total_spent,
    COUNT(t.transaction_id) AS total_transactions
FROM customers c
JOIN cards ca
    ON c.customer_id = ca.customer_id
JOIN transactions t
    ON ca.card_id = t.card_id
WHERE
    LOWER(c.segment) = 'premium'
    AND UPPER(ca.status) = 'ACTIVE'
    AND UPPER(t.transaction_status) = 'APPROVED'
    AND STRFTIME(t.transaction_date, '%Y') = '2025'
    AND t.amount > (
        SELECT AVG(amount)
        FROM transactions
        WHERE STRFTIME(transaction_date, '%Y') = '2025'
    )
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    total_spent DESC
LIMIT 20;



### Benchmark robusto

Para reducir variaciones por caché, CPU y ejecución inicial:

- realizamos ejecuciones de calentamiento;
- ejecutamos varias mediciones;
- utilizamos la **mediana** en lugar del promedio.

In [141]:
import statistics
import time


def benchmark_query(query, runs=10, warmup=2):

    # Warm-up
    for _ in range(warmup):
        con.execute(query).fetchall()

    times = []

    # Mediciones
    for _ in range(runs):

        start = time.perf_counter()

        con.execute(query).fetchall()

        elapsed = time.perf_counter() - start

        times.append(elapsed)

    return {
        "median": statistics.median(times),
        "mean": statistics.mean(times),
        "min": min(times),
        "max": max(times),
        "runs": times
    }

In [142]:
baseline_metrics = benchmark_query(slow_query)

baseline_time = baseline_metrics["median"]

print(f"Median : {baseline_metrics['median']:.4f} s")
print(f"Mean   : {baseline_metrics['mean']:.4f} s")
print(f"Min    : {baseline_metrics['min']:.4f} s")
print(f"Max    : {baseline_metrics['max']:.4f} s")

Median : 0.2541 s
Mean   : 0.2546 s
Min    : 0.2397 s
Max    : 0.2733 s


In [143]:
result = con.execute(slow_query).fetchdf()

print(f"Filas obtenidas: {len(result):,}")

result.head(10)

Filas obtenidas: 20


,customer_id,customer_name,total_spent,total_transactions
0,23550,Customer_23550,115501.0,77
1,35420,Customer_35420,114477.0,77
2,980,Customer_980,113830.0,76
3,44560,Customer_44560,113313.0,77
4,32810,Customer_32810,112525.0,74
5,31500,Customer_31500,112450.0,75
6,17490,Customer_17490,112383.0,73
7,44620,Customer_44620,112161.0,74
8,1500,Customer_1500,111837.0,72
9,23970,Customer_23970,111023.0,73


## 5. Prompting

Primero utilizaremos el enfoque tradicional de **Prompting**.

Enviaremos la query al modelo una sola vez y obtendremos una propuesta de optimización.

Usaremos **Mistral ejecutándose localmente mediante Ollama**, utilizando una API compatible con OpenAI.

In [144]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL = "mistral"

print(f"LLM ready: {MODEL}")

LLM ready: mistral


### Validación del modelo

Realizamos una llamada simple para comprobar que el notebook puede comunicarse con Mistral.

In [145]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Responde únicamente: OK"
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

 OK


### Optimización con Prompting

Enviamos la query al modelo una sola vez.

El modelo propondrá una versión optimizada, pero todavía no existe ningún ciclo de validación o reintento.

In [146]:
prompt = f"""
You are a SQL performance expert.

Optimize the following DuckDB query.

Requirements:
- Preserve exactly the same result.
- Return only the optimized SQL.
- Do not use markdown.
- Do not explain the answer.

SQL:

{slow_query}
"""

In [147]:
import re

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

raw_response = response.choices[0].message.content.strip()

# Extraer únicamente el bloque SQL si el modelo devuelve Markdown
match = re.search(r"```(?:sql)?\s*(.*?)```", raw_response, re.DOTALL | re.IGNORECASE)

optimized_query_prompt = (
    match.group(1).strip()
    if match
    else raw_response
)

print(optimized_query_prompt)

WITH avg_amount AS (
    SELECT AVG(amount) as avg_2025_amount
    FROM transactions
    WHERE STRFTIME(transaction_date, '%Y') = '2025'
)
SELECT
    c.customer_id,
    c.customer_name,
    SUM(t.amount) AS total_spent,
    COUNT(t.transaction_id) AS total_transactions
FROM customers c
JOIN cards ca
    ON c.customer_id = ca.customer_id
JOIN transactions t
    ON ca.card_id = t.card_id
WHERE
    LOWER(c.segment) = 'premium'
    AND UPPER(ca.status) = 'ACTIVE'
    AND UPPER(t.transaction_status) = 'APPROVED'
    AND STRFTIME(t.transaction_date, '%Y') = '2025'
    AND t.amount > (SELECT avg_2025_amount FROM avg_amount)
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    total_spent DESC
LIMIT 20;


### Validación de la propuesta

Medimos el rendimiento de la query propuesta y verificamos que devuelva el mismo resultado que la original.

In [148]:
prompt_metrics = benchmark_query(optimized_query_prompt)

prompt_time = prompt_metrics["median"]

improvement_prompt = (
    (baseline_time - prompt_time)
    / baseline_time
) * 100

print(f"Baseline median : {baseline_time:.4f} s")
print(f"Prompt median   : {prompt_time:.4f} s")
print(f"Improvement     : {improvement_prompt:.2f}%")

Baseline median : 0.2541 s
Prompt median   : 0.2575 s
Improvement     : -1.34%


## 6. Looping

Ahora cambiamos el enfoque: el objetivo ya no es simplemente generar una query optimizada.

El sistema ejecutará, medirá y evaluará cada propuesta. Si no alcanza el objetivo, utilizará el resultado como feedback para generar un nuevo intento.

In [149]:
TARGET_IMPROVEMENT = 0.10  # 10%
MAX_ITERATIONS = 5

print(f"Target improvement : {TARGET_IMPROVEMENT:.0%}")
print(f"Max iterations     : {MAX_ITERATIONS}")
print(f"Baseline           : {baseline_time:.4f} s")

Target improvement : 10%
Max iterations     : 5
Baseline           : 0.2541 s


### Loop de optimización

Cada iteración sigue el mismo ciclo:

**Proponer → Ejecutar → Validar → Medir → Evaluar → Reintentar**

In [150]:
import re

def extract_sql(response):
    response = response.strip()

    match = re.search(
        r"```(?:sql)?\s*(.*?)```",
        response,
        re.DOTALL | re.IGNORECASE
    )

    return match.group(1).strip() if match else response

In [151]:
# ============================================================
# LOOPING: Optimize → Execute → Validate → Measure → Evaluate
# ============================================================

def validate_results(original_query, candidate_query):
    """
    Valida que ambas queries produzcan el mismo resultado
    """
    original = con.execute(original_query).fetchall()
    candidate = con.execute(candidate_query).fetchall()

    return sorted(original) == sorted(candidate)

In [152]:
current_query = slow_query
best_query = slow_query
best_time = baseline_time

history = []
last_feedback = "No previous attempt."


for iteration in range(1, MAX_ITERATIONS + 1):

    print(f"\n{'=' * 50}")
    print(f"ITERATION {iteration}")
    print("=" * 50)

    feedback = f"""
        Baseline execution time: {baseline_time:.6f} seconds.
        Best execution time so far: {best_time:.6f} seconds.

        Goal:
        Improve execution time by at least
        {TARGET_IMPROVEMENT:.0%} compared with the baseline.

        Feedback from previous attempt:
        {last_feedback}

        Try a different valid optimization strategy.
    """

    prompt_loop = f"""
        You are a SQL performance expert.

        Optimize the following DuckDB query.

        Requirements:
        - Preserve exactly the same result.
        - Improve execution performance.
        - Return only valid DuckDB SQL.
        - Return only SQL.
        - Do not use markdown.
        - Do not explain the answer.

        SQL:

        {current_query}

        FEEDBACK:

        {feedback}
    """

    # 1. PROPOSE
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt_loop
            }
        ],
        temperature=0
    )

    candidate_query = extract_sql(
        response.choices[0].message.content
    )

    # 2. EXECUTE + VALIDATE
    try:

        candidate_metrics = benchmark_query(candidate_query)

        candidate_time = candidate_metrics["median"]

        same_result = validate_results(
            slow_query,
            candidate_query
        )

    except Exception as e:

        last_feedback = (
            "The previous SQL was invalid.\n"
            f"DuckDB error:\n{str(e)}\n"
            "Fix the SQL and do not repeat the same error."
        )

        print(f"Invalid SQL: {e}")
        continue

    # 3. EVALUATE
    improvement = (
        baseline_time - candidate_time
    ) / baseline_time

    history.append({
        "iteration": iteration,
        "time": candidate_time,
        "improvement": improvement,
        "valid": same_result
    })

    print(f"Time        : {candidate_time:.4f} s")
    print(f"Improvement : {improvement:.2%}")
    print(f"Same result : {same_result}")

    # 4. UPDATE FEEDBACK
    if not same_result:

        last_feedback = (
            "The query executed, but the result was not equivalent "
            "to the original query. Preserve the exact same result."
        )

    else:

        last_feedback = (
            f"The query was valid and preserved the result.\n"
            f"Execution time: {candidate_time:.6f} seconds.\n"
            f"Improvement: {improvement:.2%}.\n"
            f"Target: {TARGET_IMPROVEMENT:.0%}.\n"
            "The target was not reached. Try a different optimization."
        )

    # 5. SAVE BEST
    if same_result and candidate_time < best_time:

        best_time = candidate_time
        best_query = candidate_query

    # 6. STOP CONDITION
    if same_result and improvement >= TARGET_IMPROVEMENT:

        print("\nGOAL ACHIEVED")
        break

    current_query = best_query

else:

    print("\nMAX ITERATIONS REACHED")


ITERATION 1
Time        : 0.2520 s
Improvement : 0.82%
Same result : True

ITERATION 2
Invalid SQL: Parser Error: Wrong number of arguments provided to DATE function

ITERATION 3
Time        : 0.1155 s
Improvement : 54.52%
Same result : True

GOAL ACHIEVED


### Validación final

Antes de aceptar la mejor solución encontrada, verificamos nuevamente que la query sea ejecutable y produzca el mismo resultado que la consulta original.

In [153]:
print("Validating BEST QUERY...")

try:
    best_result = con.execute(best_query).fetchall()
    original_result = con.execute(slow_query).fetchall()

    same_result_final = (
        sorted(best_result) == sorted(original_result)
    )

    print("SQL valid   : True")
    print("Same result :", same_result_final)

except Exception as e:
    print("SQL valid   : False")
    print("Same result : False")
    print("Error       :", e)

Validating BEST QUERY...
SQL valid   : True
Same result : True


In [154]:
print("\n" + "=" * 60)
print("ORIGINAL QUERY")
print("=" * 60)
print(slow_query.strip())


print("\n" + "=" * 60)
print("OPTIMIZED QUERY")
print("=" * 60)
print(best_query.strip())


print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)

improvement = (
    (baseline_time - best_time)
    / baseline_time
)

print(f"Baseline       : {baseline_time:.4f} s")
print(f"Optimized      : {best_time:.4f} s")
print(f"Improvement    : {improvement:.2%}")
print(f"Goal           : {TARGET_IMPROVEMENT:.0%}")
print(f"Goal achieved  : {improvement >= TARGET_IMPROVEMENT}")


ORIGINAL QUERY
SELECT
    c.customer_id,
    c.customer_name,
    SUM(t.amount) AS total_spent,
    COUNT(t.transaction_id) AS total_transactions
FROM customers c
JOIN cards ca
    ON c.customer_id = ca.customer_id
JOIN transactions t
    ON ca.card_id = t.card_id
WHERE
    LOWER(c.segment) = 'premium'
    AND UPPER(ca.status) = 'ACTIVE'
    AND UPPER(t.transaction_status) = 'APPROVED'
    AND STRFTIME(t.transaction_date, '%Y') = '2025'
    AND t.amount > (
        SELECT AVG(amount)
        FROM transactions
        WHERE STRFTIME(transaction_date, '%Y') = '2025'
    )
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    total_spent DESC
LIMIT 20;

OPTIMIZED QUERY
WITH avg_2025 AS (
    SELECT AVG(amount) as avg_amount
    FROM transactions
    WHERE EXTRACT(YEAR FROM transaction_date) = 2025
)

SELECT
    c.customer_id,
    c.customer_name,
    SUM(t.amount) AS total_spent,
    COUNT(t.transaction_id) AS total_transactions
FROM customers c
JOIN cards ca
    ON c.customer_id 

### Historial del Loop

Visualizamos cada intento realizado por el sistema durante el proceso de optimización.

In [155]:
import pandas as pd

history_df = pd.DataFrame(history)

if not history_df.empty:

    history_df["time_ms"] = history_df["time"] * 1000
    history_df["improvement_pct"] = history_df["improvement"] * 100

    display(
        history_df[
            [
                "iteration",
                "time_ms",
                "improvement_pct",
                "valid"
            ]
        ].rename(
            columns={
                "iteration": "Iteration",
                "time_ms": "Time (ms)",
                "improvement_pct": "Improvement (%)",
                "valid": "Same Result"
            }
        ).round(2)
    )

else:
    print("No iterations recorded.")

,Iteration,Time (ms),Improvement (%),Same Result
0,1,251.97,0.82,True
1,3,115.55,54.52,True


In [156]:
summary = con.execute(slow_query).fetchdf()

summary

,customer_id,customer_name,total_spent,total_transactions
0,23550,Customer_23550,115501.0,77
1,35420,Customer_35420,114477.0,77
2,980,Customer_980,113830.0,76
3,44560,Customer_44560,113313.0,77
4,32810,Customer_32810,112525.0,74
5,31500,Customer_31500,112450.0,75
6,17490,Customer_17490,112383.0,73
7,44620,Customer_44620,112161.0,74
8,1500,Customer_1500,111837.0,72
9,23970,Customer_23970,111023.0,73


In [157]:
summary = con.execute(best_query).fetchdf()

summary

,customer_id,customer_name,total_spent,total_transactions
0,23550,Customer_23550,115501.0,77
1,35420,Customer_35420,114477.0,77
2,980,Customer_980,113830.0,76
3,44560,Customer_44560,113313.0,77
4,32810,Customer_32810,112525.0,74
5,31500,Customer_31500,112450.0,75
6,17490,Customer_17490,112383.0,73
7,44620,Customer_44620,112161.0,74
8,1500,Customer_1500,111837.0,72
9,23970,Customer_23970,111023.0,73
